# 03 - Accident Data Review: SMS × Dispatch Join

**目的：** 在進行 VD 資料建置之前，先確認兩份事故資料的可用範圍並完成對照。

## 分析步驟

1. 讀入交控事故簡訊紀錄表（全24個月），統計各月份「事件排除」非空率
2. 關鍵欄位完整性熱圖（哪些月份哪些欄位有資料）
3. 讀入派遣系統資料（`accident_data.csv`）
4. 資料標準化（國道、方向、時間戳記）
5. Fuzzy join：以 國道 + 方向 + 里程（±0.5 km）+ 時間（±10 min）匹配兩份資料
6. 匹配結果視覺化：整體匹配率、按月、按路線、時間差與里程差分布
7. 2023-06 ~ 2024-03 額外可用筆數統計
8. 結論與建議

In [ ]:
import os, sys, io, zipfile, re, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
from datetime import datetime, timedelta
!wget -O NotoSansTC.otf https://github.com/googlefonts/noto-cjk/raw/main/Sans/OTF/TraditionalChinese/NotoSansCJKtc-Regular.otf
import matplotlib.font_manager as fm

# 2. 強制將字體加入 matplotlib 的字體管理員中
fm.fontManager.addfont('NotoSansTC.otf')

# 3. 抓取該字體的正確名稱並設定為預設字體
# 這樣做可以確保名稱絕對正確，不用去瞎猜 matplotlib 怎麼解析字體名稱的
font_prop = fm.FontProperties(fname='NotoSansTC.otf')
plt.rc('font', family=font_prop.get_name())

# 4. 解決設定中文字體後，負號 '-' 變成方塊的問題
plt.rcParams['axes.unicode_minus'] = False
warnings.filterwarnings('ignore', category=UserWarning)

# ── environment detection ────────────────────────────────────────────────────
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    _DEFAULT_ROOT = '/content/drive/MyDrive/陽交大高速公路計畫/main'
else:
    _here = os.path.abspath(os.getcwd())
    _DEFAULT_ROOT = os.path.dirname(_here) if os.path.basename(_here) == 'notebooks' else _here

PROJECT_ROOT = Path(os.environ.get('PROJECT_ROOT', _DEFAULT_ROOT))
os.chdir(PROJECT_ROOT)
print('PROJECT_ROOT =', PROJECT_ROOT)

OUTPUT_DIR = PROJECT_ROOT / 'output' / 'accident_review'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Output dir  =', OUTPUT_DIR)

---
## 1. 讀入簡訊紀錄表（全24個月）

資料來源：`data/11206_11405交控事故簡訊紀錄表-*.zip`

ZIP 內部路徑：`11206_11405交控事故簡訊紀錄表/{year}年{month}月事故處理時間紀錄表.xlsx`

民國年換算：ROC year + 1911 = 西元年（112 → 2023, 113 → 2024, 114 → 2025）

In [ ]:
ZIP_NAME = '11206_11405交控事故簡訊紀錄表-20251125T072343Z-1-001.zip'
ZIP_PATH = PROJECT_ROOT / 'data' / ZIP_NAME

FNAME_RE = re.compile(r'(\d+)年(\d+)月')

def roc_to_ad(roc_year: int) -> int:
    return roc_year + 1911

monthly_frames = []

with zipfile.ZipFile(ZIP_PATH) as z:
    xlsx_names = [n for n in z.namelist() if n.endswith('.xlsx')]
    print(f'Found {len(xlsx_names)} xlsx files in ZIP')

    for name in sorted(xlsx_names):
        m = FNAME_RE.search(name)
        if not m:
            print(f'  SKIP (cannot parse year/month): {name}')
            continue
        roc_year = int(m.group(1))
        month    = int(m.group(2))
        year     = roc_to_ad(roc_year)

        with z.open(name) as f:
            try:
                df = pd.read_excel(io.BytesIO(f.read()))
            except Exception as e:
                print(f'  ERROR reading {name}: {e}')
                continue

        df['_year_ad'] = year
        df['_month']   = month
        df['_ym']      = f'{year}-{month:02d}'
        df['_filename'] = Path(name).name
        monthly_frames.append(df)
        print(f'  {year}-{month:02d}  rows={len(df):>4}  cols={df.shape[1]}')

sms_raw = pd.concat(monthly_frames, ignore_index=True)
print(f'\nTotal SMS rows: {len(sms_raw):,}')
print(f'Columns ({sms_raw.shape[1]}):',
      [c for c in sms_raw.columns if not c.startswith('_')])

---
## 2. 各月份「事件排除」非空率

「事件排除」是計算**事故影響時間（TIID）** 最直接的欄位（= 事件排除 − 事件發生）。
非空率 < 80% 的月份可能不適合直接使用；需確認其他時間欄位是否可替代。

In [ ]:
# ── per-month fill rate ──────────────────────────────────────────────────────
grp = sms_raw.groupby('_ym')
fillrate = pd.DataFrame({
    'ym'              : list(grp.groups.keys()),
    'total_rows'      : grp.size().values,
    'clearance_filled': grp['事件排除'].apply(lambda s: s.notna().sum()).values,
    'event_filled'    : grp['事件發生'].apply(lambda s: s.notna().sum()).values,
    'duration_filled' : grp['處理分鐘'].apply(lambda s: s.notna().sum()).values,
})
fillrate['clearance_rate'] = fillrate['clearance_filled'] / fillrate['total_rows'] * 100
fillrate['event_rate']     = fillrate['event_filled']     / fillrate['total_rows'] * 100
fillrate['duration_rate']  = fillrate['duration_filled']  / fillrate['total_rows'] * 100

print(fillrate[['ym','total_rows','clearance_filled','clearance_rate',
                 'event_filled','event_rate']].to_string(index=False))

# ── figure ───────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 9), gridspec_kw={'height_ratios': [3, 1.2]})

ax = axes[0]
yms   = fillrate['ym'].tolist()
rates = fillrate['clearance_rate'].tolist()
colors = ['#2ecc71' if r >= 80 else '#f39c12' if r >= 50 else '#e74c3c'
          for r in rates]
bars = ax.bar(range(len(yms)), rates, color=colors, edgecolor='white', linewidth=0.5)

ax.axhline(80, color='#27ae60', linewidth=1.2, linestyle='--', label='80% threshold')
ax.axhline(50, color='#e67e22', linewidth=1.2, linestyle='--', label='50% threshold')

for bar, rate in zip(bars, rates):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1.5,
            f'{rate:.0f}%', ha='center', va='bottom', fontsize=7.5, fontweight='bold')

ax.axvspan(-0.5, 6.5,  alpha=0.04, color='gray', label='2023 (no dispatch data)')  # 7 months: idx 0-6
ax.axvspan(6.5,  18.5, alpha=0.04, color='blue', label='2024 (dispatch overlap)')  # 12 months: idx 7-18
ax.axvspan(18.5, 23.5, alpha=0.04, color='green', label='2025 (dispatch overlap)') # 5 months: idx 19-23

ax.set_xticks(range(len(yms)))
ax.set_xticklabels(yms, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('事件排除 非空率 (%)')
ax.set_title('各月份「事件排除」欄填寫率', fontsize=13, fontweight='bold')
ax.set_ylim(0, 115)
ax.legend(loc='lower right', fontsize=8)
ax.yaxis.set_major_formatter(mticker.PercentFormatter())

# ── row count subplot ────────────────────────────────────────────────────────
ax2 = axes[1]
ax2.bar(range(len(yms)), fillrate['total_rows'], color='#3498db', alpha=0.7,
        edgecolor='white', linewidth=0.5)
for i, n in enumerate(fillrate['total_rows']):
    ax2.text(i, n + 3, str(n), ha='center', va='bottom', fontsize=7)
ax2.set_xticks(range(len(yms)))
ax2.set_xticklabels(yms, rotation=45, ha='right', fontsize=9)
ax2.set_ylabel('總筆數')
ax2.set_title('各月份總筆數', fontsize=11)

plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'sms_clearance_fill_rate.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {OUTPUT_DIR}/sms_clearance_fill_rate.png')

---
## 3. 關鍵欄位完整性熱圖

檢視各時間流程欄位（事件發生 → 接獲通報 → CCTV → CMS → 出發 → 抵達 → 完成 → 事件排除）
在各月份的填寫率，用顏色快速辨識哪些欄位哪些月份有資料。

In [ ]:
TIME_COLS = [
    '事件發生',
    '交控中心\n接獲通報',
    'CCTV\n監看現場',
    'CMS\n發布資訊',
    '交控中心\n通報工務段',
    '事故處理\n小組出發',
    '事故處理\n小組抵達',
    '事故處理\n小組完成',
    '事件排除',
    '處理分鐘',
]
TIME_LABELS = [
    '事件發生', '接獲通報', 'CCTV監看', 'CMS發布',
    '通報工務段', '小組出發', '小組抵達', '小組完成',
    '事件排除', '處理分鐘',
]

# build fill-rate matrix: rows = months, cols = time columns
existing_cols = [c for c in TIME_COLS if c in sms_raw.columns]
ym_list = sorted(sms_raw['_ym'].unique())
matrix = np.zeros((len(ym_list), len(existing_cols)))

for i, ym in enumerate(ym_list):
    sub = sms_raw[sms_raw['_ym'] == ym]
    for j, col in enumerate(existing_cols):
        matrix[i, j] = sub[col].notna().mean() * 100

col_labels = [TIME_LABELS[TIME_COLS.index(c)] for c in existing_cols]

fig, ax = plt.subplots(figsize=(12, 8))
im = ax.imshow(matrix, aspect='auto', cmap='RdYlGn', vmin=0, vmax=100)

ax.set_xticks(range(len(col_labels)))
ax.set_xticklabels(col_labels, rotation=30, ha='right', fontsize=10)
ax.set_yticks(range(len(ym_list)))
ax.set_yticklabels(ym_list, fontsize=9)

for i in range(len(ym_list)):
    for j in range(len(existing_cols)):
        val = matrix[i, j]
        color = 'black' if 20 < val < 80 else 'white'
        ax.text(j, i, f'{val:.0f}', ha='center', va='center',
                fontsize=7.5, color=color, fontweight='bold')

cbar = fig.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label('填寫率 (%)', fontsize=10)

# Highlight dispatch-overlap rows (2024-04 onwards)
try:
    overlap_start = ym_list.index('2024-04')
    ax.axhline(overlap_start - 0.5, color='dodgerblue', linewidth=2,
               linestyle='--', label='派遣資料起始 (2024-04)')
    ax.legend(fontsize=9, loc='upper right')
except ValueError:
    pass

ax.set_title('各月份關鍵時間欄位填寫率 (%)', fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'sms_column_fill_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {OUTPUT_DIR}/sms_column_fill_heatmap.png')

---
## 4. 讀入派遣系統資料

`data/accident_data.csv`：由 `process_accident_data.py` 產出，包含 2024-04 ~ 2025-05 的事故記錄（篩選後 2043 筆）。

In [ ]:
dispatch_raw = pd.read_csv(
    PROJECT_ROOT / 'data' / 'accident_data.csv',
    encoding='utf-8-sig',
    parse_dates=['發送', '到達', '處理完成'],
    dayfirst=False,
)
print(f'Dispatch rows: {len(dispatch_raw):,}')
print(f'Date range: {dispatch_raw["發送"].min().date()} → {dispatch_raw["發送"].max().date()}')
print('Freeway dist:')
print(dispatch_raw['國道'].value_counts().to_string())
print('\nDirection dist:')
print(dispatch_raw['方向'].value_counts().to_string())

---
## 5. 資料標準化

對兩份資料建立統一的 key 欄位：

| 欄位 | SMS 原始 | 標準化後 | Dispatch 對應欄 |
|---|---|---|---|
| 國道 | `國道1號` | `N1` | `國道`（已標準化） |
| 方向 | `南向` | `南` | `方向` |
| 里程 | `里程`（km float） | 同 | `里程數` |
| 事件時間 | 年/月/日 + `事件發生` HH:MM:SS | `event_ts` | `發送` datetime |

> 注意：SMS 的 年/月/日/時/分 欄位對應 **事件排除** 的時間，而非事件發生時間。
> 事件發生時間需從 `年/月/日`（日期）加上 `事件發生`（HH:MM:SS 字串）合成。

In [ ]:
# ── SMS standardization ──────────────────────────────────────────────────────
HW_MAP = {'國道1號': 'N1', '國道3號': 'N3', '國道4號': 'N4', '國道6號': 'N6'}

def parse_sms_time(year_ad, month, day, time_str):
    """Combine date parts and HH:MM:SS string into a datetime.
    Returns NaT if time_str is null or unparseable.
    """
    if pd.isna(time_str):
        return pd.NaT
    try:
        t = str(time_str).strip()
        parts = t.split(':')
        h, mi, s = int(parts[0]), int(parts[1]), int(float(parts[2]))
        base = datetime(int(year_ad), int(month), int(day))
        dt = base + timedelta(hours=h, minutes=mi, seconds=s)
        return dt
    except Exception:
        return pd.NaT

sms = sms_raw.copy()
sms['freeway']   = sms['國道名稱'].map(HW_MAP)
sms['direction'] = sms['方向'].str.replace('向', '', regex=False)
sms['mileage']   = pd.to_numeric(sms['里程'], errors='coerce')

sms['event_ts'] = [
    parse_sms_time(r['_year_ad'], r['_month'], r['日'], r['事件發生'])
    for _, r in sms.iterrows()
]
sms['clearance_ts'] = [
    parse_sms_time(r['_year_ad'], r['_month'], r['日'], r['事件排除'])
    for _, r in sms.iterrows()
]

# event minutes since epoch (for arithmetic)
epoch = pd.Timestamp('2023-01-01')
sms['event_min'] = (sms['event_ts'] - epoch).dt.total_seconds() / 60
sms['has_clearance'] = sms['clearance_ts'].notna()
sms['duration_min']  = (
    (sms['clearance_ts'] - sms['event_ts']).dt.total_seconds() / 60
).where(sms['has_clearance'])

print('SMS standardization complete.')
print(f'  freeway coverage  : {sms["freeway"].value_counts().to_dict()}')
print(f'  direction coverage: {sms["direction"].value_counts().to_dict()}')
print(f'  event_ts non-null : {sms["event_ts"].notna().sum():,} / {len(sms):,}')
print(f'  has_clearance     : {sms["has_clearance"].sum():,} / {len(sms):,}')

# ── Dispatch standardization ─────────────────────────────────────────────────
dispatch = dispatch_raw.copy()
dispatch['dispatch_min'] = (dispatch['發送'] - epoch).dt.total_seconds() / 60
dispatch['tiid_min'] = (
    (dispatch['處理完成'] - dispatch['發送']).dt.total_seconds() / 60
).where(dispatch['處理完成'].notna())

print(f'\nDispatch standardization complete.')
print(f'  rows: {len(dispatch):,}  |  tiid non-null: {dispatch["tiid_min"].notna().sum():,}')

---
## 6. Fuzzy Join

匹配條件（AND）：
- `freeway` 相同（N1 / N3 / N4 / N6）
- `direction` 相同（南 / 北 / 東 / 西）
- `|里程差| ≤ 0.5 km`
- `|事件發生 − 派遣發送| ≤ 10 分鐘`

若同一 SMS 記錄有多個 dispatch 候選，取時間差最小者。

僅對 2024-04 起的 SMS 資料執行 join（dispatch 資料從此時開始）。

In [ ]:
MILE_TOL = 0.5   # km
TIME_TOL = 10.0  # minutes

# Filter SMS to overlap period
sms_overlap = sms[
    sms['event_ts'].notna() &
    (sms['event_ts'] >= pd.Timestamp('2024-04-01'))
].copy()
print(f'SMS records in overlap period (2024-04+): {len(sms_overlap):,}')

match_records = []

for (fw, dr), sms_grp in sms_overlap.groupby(['freeway', 'direction']):
    disp_grp = dispatch[
        (dispatch['國道'] == fw) & (dispatch['方向'] == dr)
    ]
    if len(disp_grp) == 0:
        for idx in sms_grp.index:
            match_records.append({'sms_idx': idx, 'disp_idx': None,
                                   'matched': False, 'mile_diff': None,
                                   'time_diff_min': None})
        continue

    sms_mile = sms_grp['mileage'].values        # (n_sms,)
    sms_tmin = sms_grp['event_min'].values      # (n_sms,)
    disp_mile = disp_grp['里程數'].values        # (n_disp,)
    disp_tmin = disp_grp['dispatch_min'].values  # (n_disp,)

    # Vectorized pairwise diffs: (n_sms, n_disp)
    mile_diff = np.abs(sms_mile[:, None] - disp_mile[None, :])
    time_diff = np.abs(sms_tmin[:, None] - disp_tmin[None, :])

    candidate = (mile_diff <= MILE_TOL) & (time_diff <= TIME_TOL)

    for i, sms_idx in enumerate(sms_grp.index):
        cands = np.where(candidate[i])[0]
        if len(cands) > 0:
            best = cands[np.argmin(time_diff[i, cands])]
            match_records.append({
                'sms_idx'      : sms_idx,
                'disp_idx'     : disp_grp.index[best],
                'matched'      : True,
                'mile_diff'    : mile_diff[i, best],
                'time_diff_min': time_diff[i, best],
            })
        else:
            match_records.append({
                'sms_idx'      : sms_idx,
                'disp_idx'     : None,
                'matched'      : False,
                'mile_diff'    : None,
                'time_diff_min': None,
            })

match_df = pd.DataFrame(match_records)
n_matched = match_df['matched'].sum()
n_total   = len(match_df)
match_rate = n_matched / n_total * 100

print(f'\n=== Fuzzy Join Results ===')
print(f'SMS records in overlap   : {n_total:,}')
print(f'Matched to dispatch      : {n_matched:,}')
print(f'Unmatched                : {n_total - n_matched:,}')
print(f'Match rate               : {match_rate:.1f}%  (target ≥ 80%)')
status = '✓ PASS' if match_rate >= 80 else '✗ BELOW TARGET'
print(f'Status                   : {status}')

---
## 7. 匹配結果視覺化

In [ ]:
# Attach metadata to match_df for grouping
match_df = match_df.join(sms_overlap[['_ym', 'freeway', 'direction']].rename(
    columns={'_ym': 'ym', 'freeway': 'fw', 'direction': 'dir'}
), on='sms_idx')

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# ── (A) Match rate by month ──────────────────────────────────────────────────
ax = axes[0, 0]
by_ym = match_df.groupby('ym')['matched'].agg(['sum', 'count'])
by_ym['rate'] = by_ym['sum'] / by_ym['count'] * 100
colors_ym = ['#2ecc71' if r >= 80 else '#f39c12' if r >= 50 else '#e74c3c'
             for r in by_ym['rate']]
bars = ax.bar(range(len(by_ym)), by_ym['rate'], color=colors_ym,
              edgecolor='white', linewidth=0.5)
ax.axhline(80, color='#27ae60', linewidth=1.2, linestyle='--')
for bar, (_, row) in zip(bars, by_ym.iterrows()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1.5,
            f'{row["rate"]:.0f}%', ha='center', va='bottom', fontsize=8)
ax.set_xticks(range(len(by_ym)))
ax.set_xticklabels(by_ym.index.tolist(), rotation=45, ha='right', fontsize=8)
ax.set_ylabel('匹配率 (%)')
ax.set_title('各月份匹配率（SMS ↔ Dispatch）', fontweight='bold')
ax.set_ylim(0, 110)
ax.yaxis.set_major_formatter(mticker.PercentFormatter())

# ── (B) Match rate by freeway × direction ────────────────────────────────────
ax = axes[0, 1]
match_df['fw_dir'] = match_df['fw'] + '-' + match_df['dir']
by_fwdir = match_df.groupby('fw_dir')['matched'].agg(['sum', 'count'])
by_fwdir['rate'] = by_fwdir['sum'] / by_fwdir['count'] * 100
colors_fwd = ['#2ecc71' if r >= 80 else '#f39c12' if r >= 50 else '#e74c3c'
              for r in by_fwdir['rate']]
bars2 = ax.bar(range(len(by_fwdir)), by_fwdir['rate'], color=colors_fwd,
               edgecolor='white', linewidth=0.5)
ax.axhline(80, color='#27ae60', linewidth=1.2, linestyle='--')
for bar, (_, row) in zip(bars2, by_fwdir.iterrows()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1.5,
            f'{row["rate"]:.0f}%\n(n={int(row["count"])})',
            ha='center', va='bottom', fontsize=8)
ax.set_xticks(range(len(by_fwdir)))
ax.set_xticklabels(by_fwdir.index.tolist(), fontsize=9)
ax.set_ylabel('匹配率 (%)')
ax.set_title('各路線匹配率', fontweight='bold')
ax.set_ylim(0, 115)
ax.yaxis.set_major_formatter(mticker.PercentFormatter())

# ── (C) Time diff distribution ───────────────────────────────────────────────
ax = axes[1, 0]
matched_only = match_df[match_df['matched']]
ax.hist(matched_only['time_diff_min'], bins=30, color='#3498db',
        edgecolor='white', linewidth=0.4)
ax.axvline(matched_only['time_diff_min'].median(), color='red',
           linestyle='--', linewidth=1.5,
           label=f'Median = {matched_only["time_diff_min"].median():.1f} min')
ax.set_xlabel('|SMS 事件發生 − Dispatch 發送| (分鐘)')
ax.set_ylabel('筆數')
ax.set_title('匹配對的時間差分布', fontweight='bold')
ax.legend(fontsize=9)

# ── (D) Mileage diff distribution ───────────────────────────────────────────
ax = axes[1, 1]
ax.hist(matched_only['mile_diff'], bins=25, color='#9b59b6',
        edgecolor='white', linewidth=0.4)
ax.axvline(matched_only['mile_diff'].median(), color='red',
           linestyle='--', linewidth=1.5,
           label=f'Median = {matched_only["mile_diff"].median():.2f} km')
ax.set_xlabel('|里程差| (km)')
ax.set_ylabel('筆數')
ax.set_title('匹配對的里程差分布', fontweight='bold')
ax.legend(fontsize=9)

fig.suptitle(
    f'Fuzzy Join 結果  |  總匹配率 = {match_rate:.1f}%  '
    f'({n_matched:,}/{n_total:,})  |  容許值: 里程 ±{MILE_TOL}km, 時間 ±{TIME_TOL}min',
    fontsize=12, fontweight='bold', y=1.01
)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'sms_dispatch_join_results.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {OUTPUT_DIR}/sms_dispatch_join_results.png')

### 7b. 匹配對的時間差 vs 里程差 散點圖

確認匹配的品質分布——理想情況下大多數點應集中在左下角（時間差小、里程差小）。

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(
    matched_only['time_diff_min'],
    matched_only['mile_diff'],
    alpha=0.35, s=12, c='#2980b9'
)
ax.axvline(TIME_TOL, color='red',   linestyle='--', linewidth=1, label=f'時間容許 {TIME_TOL} min')
ax.axhline(MILE_TOL, color='orange', linestyle='--', linewidth=1, label=f'里程容許 {MILE_TOL} km')
ax.set_xlabel('時間差 (分鐘)', fontsize=11)
ax.set_ylabel('里程差 (km)', fontsize=11)
ax.set_title('匹配對品質散點圖（時間差 vs 里程差）', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)

# Density annotation in corners
tight = ((matched_only['time_diff_min'] <= 3) & (matched_only['mile_diff'] <= 0.2)).sum()
ax.text(0.98, 0.98, f'時間≤3min & 里程≤0.2km:\n{tight:,} 筆 ({tight/len(matched_only)*100:.1f}%)',
        transform=ax.transAxes, ha='right', va='top', fontsize=9,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))

plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'sms_dispatch_match_quality.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {OUTPUT_DIR}/sms_dispatch_match_quality.png')

---
## 8. TIID 比較：SMS 事件排除 vs Dispatch 處理完成

對成功匹配的記錄，比較兩種目標變數的數值差異：
- **SMS TIID**：`事件排除 − 事件發生`（min）— 車道恢復通行
- **Dispatch TIID**：`處理完成 − 發送`（min）— 救援完成

兩者語意不同，SMS TIID 更接近「真實交通影響時間」。

In [ ]:
# Build matched pairs table
matched_sms  = sms_overlap.loc[matched_only['sms_idx'].values].reset_index(drop=True)
matched_disp = dispatch.loc[matched_only['disp_idx'].values].reset_index(drop=True)

pairs = pd.DataFrame({
    'ym'           : matched_sms['_ym'],
    'freeway'      : matched_sms['freeway'],
    'mileage'      : matched_sms['mileage'],
    'sms_tiid'     : matched_sms['duration_min'].values,
    'dispatch_tiid': matched_disp['tiid_min'].values,
    'time_diff'    : matched_only['time_diff_min'].values,
    'mile_diff'    : matched_only['mile_diff'].values,
})
pairs_valid = pairs.dropna(subset=['sms_tiid', 'dispatch_tiid'])
print(f'Matched pairs with both TIIDs: {len(pairs_valid):,}')

if len(pairs_valid) > 10:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # Scatter: SMS TIID vs Dispatch TIID
    ax = axes[0]
    lim = max(pairs_valid['sms_tiid'].max(), pairs_valid['dispatch_tiid'].max()) * 1.05
    ax.scatter(pairs_valid['dispatch_tiid'], pairs_valid['sms_tiid'],
               alpha=0.3, s=14, c='#16a085')
    ax.plot([0, lim], [0, lim], 'r--', linewidth=1, label='y = x')
    ax.set_xlabel('Dispatch TIID (處理完成 − 發送) [min]', fontsize=10)
    ax.set_ylabel('SMS TIID (事件排除 − 事件發生) [min]', fontsize=10)
    ax.set_title('兩種 TIID 的對照', fontweight='bold')
    ax.legend(fontsize=9)
    ax.set_xlim(0, lim); ax.set_ylim(0, lim)

    diff = pairs_valid['sms_tiid'] - pairs_valid['dispatch_tiid']
    ax.text(0.02, 0.98,
            f'Corr = {pairs_valid["sms_tiid"].corr(pairs_valid["dispatch_tiid"]):.3f}\n'
            f'Mean diff = {diff.mean():.1f} min\n'
            f'Median diff = {diff.median():.1f} min',
            transform=ax.transAxes, va='top', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

    # Distribution of diff
    ax2 = axes[1]
    ax2.hist(diff.clip(-60, 120), bins=40, color='#8e44ad', edgecolor='white', linewidth=0.4)
    ax2.axvline(0,    color='red',  linestyle='--', linewidth=1.2, label='zero diff')
    ax2.axvline(diff.median(), color='blue', linestyle='--', linewidth=1.2,
                label=f'Median = {diff.median():.1f} min')
    ax2.set_xlabel('SMS TIID − Dispatch TIID (分鐘)', fontsize=10)
    ax2.set_ylabel('筆數')
    ax2.set_title('TIID 差值分布 (SMS − Dispatch)', fontweight='bold')
    ax2.legend(fontsize=9)

    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / 'sms_vs_dispatch_tiid.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved → {OUTPUT_DIR}/sms_vs_dispatch_tiid.png')
else:
    print('Not enough matched pairs with both TIIDs to plot.')

---
## 9. 2023-06 ~ 2024-03 額外可用筆數

這段期間沒有派遣資料，SMS 是唯一的事故來源。
統計其中「事件排除」有值的筆數（= 可計算 TIID 的記錄）。

In [ ]:
sms_early = sms[
    sms['event_ts'].notna() &
    (sms['event_ts'] < pd.Timestamp('2024-04-01'))
].copy()

early_stats = sms_early.groupby('_ym').agg(
    total=('event_ts', 'count'),
    has_clearance=('has_clearance', 'sum'),
    has_duration=('duration_min', lambda x: x.notna().sum()),
).reset_index()
early_stats['clearance_rate'] = early_stats['has_clearance'] / early_stats['total'] * 100

print('2023-06 ~ 2024-03 SMS records (pre-dispatch era):')
print(early_stats.to_string(index=False))
print(f'\nTotal rows            : {early_stats["total"].sum():,}')
print(f'With 事件排除         : {early_stats["has_clearance"].sum():,}')
print(f'With computable TIID  : {early_stats["has_duration"].sum():,}')

fig, ax = plt.subplots(figsize=(10, 4))
x = range(len(early_stats))
bars_total = ax.bar(x, early_stats['total'], label='總筆數', color='#bdc3c7', width=0.6)
bars_ok    = ax.bar(x, early_stats['has_clearance'], label='有事件排除', color='#27ae60', width=0.6)
ax.set_xticks(list(x))
ax.set_xticklabels(early_stats['_ym'].tolist(), rotation=45, ha='right')
ax.set_ylabel('筆數')
ax.set_title('2023-06 ~ 2024-03 各月份可用筆數（無派遣資料對照）', fontweight='bold')
ax.legend()
for xi, (tot, ok) in enumerate(zip(early_stats['total'], early_stats['has_clearance'])):
    ax.text(xi, tot + 2, str(tot), ha='center', va='bottom', fontsize=8, color='gray')
    ax.text(xi, ok + 2, str(ok),  ha='center', va='bottom', fontsize=8, color='#27ae60')
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'sms_early_period.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {OUTPUT_DIR}/sms_early_period.png')

---
## 10. 結論與建議

In [ ]:
# Compute key numbers for the summary
total_sms          = len(sms)
sms_with_clearance = sms['has_clearance'].sum()
sms_with_tiid      = sms['duration_min'].notna().sum()
early_usable       = early_stats['has_clearance'].sum() if len(early_stats) > 0 else 0

# months above 80% threshold
good_months = (fillrate['clearance_rate'] >= 80).sum()
all_months  = len(fillrate)

summary_lines = [
    '=' * 60,
    '  SUMMARY',
    '=' * 60,
    f'  SMS 資料總筆數         : {total_sms:,} ({all_months} 個月)',
    f'  事件排除 非空筆數      : {sms_with_clearance:,}',
    f'  可計算 TIID 筆數       : {sms_with_tiid:,}',
    f'  填寫率 ≥ 80% 月份數    : {good_months}/{all_months}',
    '',
    f'  Fuzzy join 匹配率      : {match_rate:.1f}%  (target ≥ 80%)',
    f'  匹配筆數               : {n_matched:,} / {n_total:,}',
    '',
    f'  2023-06~2024-03 可用   : {early_usable:,} 筆（SMS 獨立來源）',
    '',
    '  建議：',
]

recs = []
if match_rate >= 80:
    recs.append('  [✓] 匹配率達標，可用 SMS 事件排除時間作為 TIID 目標變數')
    recs.append('  [✓] 對派遣資料補充 SMS 的更細欄位（車道佔用、事故標記）')
else:
    recs.append(f'  [!] 匹配率 {match_rate:.1f}% 未達 80%，建議：')
    recs.append('       - 放寬時間容許值至 ±15 min 後再評估')
    recs.append('       - 確認 SMS 事件發生欄位是否有系統性偏移')

if good_months < all_months:
    bad = fillrate[fillrate['clearance_rate'] < 80]['ym'].tolist()
    recs.append(f'  [!] 以下月份填寫率 < 80%，建議排除或特別標記：')
    recs.append(f'       {bad}')

recs.append('  [→] 下一步：執行 process_sms_data.py 整合 SMS 至 accident_data.csv')
recs.append('  [→] 再執行 01_build_data_layers.ipynb 補建 2023-06 起的 VD 資料')

for line in summary_lines + recs + ['=' * 60]:
    print(line)

# Save summary as text
(OUTPUT_DIR / 'sms_dispatch_summary.txt').write_text(
    '\n'.join(summary_lines + recs + ['=' * 60]),
    encoding='utf-8'
)
print(f'\nSummary saved → {OUTPUT_DIR}/sms_dispatch_summary.txt')